In [1]:
!pip install pandas matplotlib numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

!wget -q -O /tmp/SourceHanSerifTW-VF.ttf \
  https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf

fm.fontManager.addfont('/tmp/SourceHanSerifTW-VF.ttf')
mpl.rc('font', family='Source Han Serif TW VF')
plt.rcParams['axes.unicode_minus'] = False

print('字體設定完成，可用名稱：',
      [f.name for f in fm.fontManager.ttflist if 'Source Han' in f.name])

字體設定完成，可用名稱： ['Source Han Serif TW VF']


## import 套件

In [3]:
"""
TDCC 股權分散表 CSV 分析腳本
-------------------------------
用法：
    python tdcc_from_csv.py TDCC_OD_1-5.csv 0050
    python tdcc_from_csv.py TDCC_OD_1-5.csv 2330 20260522   # 指定日期

取得 CSV：https://opendata.tdcc.com.tw/getOD.ashx?id=1-5
安裝依賴：pip install pandas matplotlib numpy
"""

import sys
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from matplotlib import rcParams
from matplotlib.patches import Patch

# 讀取、篩選、繪圖

In [5]:
# ── 持股分級對應區間標籤 ──────────────────────────────────────────────────────
GRADE_LABEL = {
    1:  '1-999',
    2:  '1,000-5,000',
    3:  '5,001-10,000',
    4:  '10,001-15,000',
    5:  '15,001-20,000',
    6:  '20,001-30,000',
    7:  '30,001-40,000',
    8:  '40,001-50,000',
    9:  '50,001-100,000',
    10: '100,001-200,000',
    11: '200,001-400,000',
    12: '400,001-600,000',
    13: '600,001-800,000',
    14: '800,001-1,000,000',
    15: '1,000,001以上',
}

# ── 讀取 & 篩選 ───────────────────────────────────────────────────────────────
def load_data(csv_path: str, stock_no: str, date: str = None) -> tuple[pd.DataFrame, str]:
    df = pd.read_csv(csv_path, encoding='utf-8', dtype={'證券代號': str})
    df['證券代號'] = df['證券代號'].str.strip()
    df['資料日期']  = df['資料日期'].astype(str).str.strip()

    # 篩股號
    sub = df[df['證券代號'] == stock_no.strip()]
    if sub.empty:
        raise ValueError(f'找不到股票代號 {stock_no}，請確認代號是否正確')

    # 篩日期
    available_dates = sorted(sub['資料日期'].unique(), reverse=True)
    if date is None:
        date = available_dates[0]
        print(f'使用最新日期：{date}（可用日期：{", ".join(available_dates[:5])}…）')
    elif date not in available_dates:
        raise ValueError(f'日期 {date} 無資料，可用：{available_dates}')

    sub = sub[sub['資料日期'] == date].copy()

    # 只留分級 1–15（排除零股 16 和合計 17）
    sub = sub[sub['持股分級'].between(1, 15)].copy()
    sub['持股區間'] = sub['持股分級'].map(GRADE_LABEL)

    # 計算累計欄位
    sub = sub.sort_values('持股分級').reset_index(drop=True)
    total_people = sub['人數'].sum()
    total_shares = sub['股數'].sum()
    sub['人數佔比'] = sub['人數'] / total_people * 100
    sub['股數佔比'] = sub['占集保庫存數比例%']
    sub['人數累計'] = sub['人數佔比'].cumsum()
    sub['股數累計'] = sub['股數佔比'].cumsum()
    sub['平均持股']  = sub['股數'] / sub['人數']

    return sub, date

# ── 繪圖 ──────────────────────────────────────────────────────────────────────
def plot_analysis(df: pd.DataFrame, stock_no: str, date: str, out_path: str):
    #rcParams['font.family'] = ['Noto Sans CJK JP', 'Microsoft JhengHei',
    #                           'PingFang TC', 'Apple LiGothic', 'DejaVu Sans', 'sans-serif']
    #plt.rcParams['axes.unicode_minus'] = False
    rcParams['font.family'] = ['Source Han Serif TW VF']

    BLUE   = '#1a73e8'; ORANGE = '#fa7c17'; GREEN  = '#34a853'
    RED    = '#ea4335'; PURPLE = '#9c27b0'; GRAY   = '#5f6368'
    BG     = '#f8f9fa'; CARD   = '#ffffff'

    title_date = f'{date[:4]}/{date[4:6]}/{date[6:]}'
    fig = plt.figure(figsize=(20, 24), facecolor=BG)
    fig.suptitle(f'{stock_no}  受益人持股分布分析  （資料日期：{title_date}）',
                 fontsize=22, fontweight='bold', color='#202124', y=0.98)
    gs = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35,
                  left=0.07, right=0.96, top=0.94, bottom=0.04)

    labels = df['持股區間'].tolist()
    x = range(len(df))

    # ① 人數分布
    ax1 = fig.add_subplot(gs[0, 0]); ax1.set_facecolor(CARD)
    bars = ax1.bar(x, df['人數'] / 10000, color=BLUE, edgecolor='white', linewidth=0.5)
    ax1.set_title('① 各持股區間人數分布', fontsize=14, fontweight='bold', pad=10)
    ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax1.set_ylabel('人數（萬人）', fontsize=10)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}萬'))
    for bar in sorted(bars, key=lambda b: b.get_height(), reverse=True)[:3]:
        ax1.annotate(f'{bar.get_height():.1f}萬',
                     xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                     xytext=(0, 4), textcoords='offset points',
                     ha='center', fontsize=8, color=BLUE, fontweight='bold')
    ax1.grid(axis='y', linestyle='--', alpha=0.5)
    ax1.spines[['top','right']].set_visible(False)

    # ② 股數佔比
    ax2 = fig.add_subplot(gs[0, 1]); ax2.set_facecolor(CARD)
    colors2 = [RED if v > 10 else ORANGE if v > 5 else GREEN for v in df['股數佔比']]
    bars2 = ax2.bar(x, df['股數佔比'], color=colors2, edgecolor='white', linewidth=0.5)
    ax2.set_title('② 各持股區間股數佔比', fontsize=14, fontweight='bold', pad=10)
    ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax2.set_ylabel('佔集保庫存數 (%)', fontsize=10)
    for bar in bars2:
        if bar.get_height() > 3:
            ax2.annotate(f'{bar.get_height():.1f}%',
                         xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                         xytext=(0, 4), textcoords='offset points',
                         ha='center', fontsize=8, fontweight='bold')
    ax2.legend(handles=[Patch(facecolor=RED, label='>10%'),
                        Patch(facecolor=ORANGE, label='5-10%'),
                        Patch(facecolor=GREEN, label='<5%')],
               fontsize=8, loc='upper right')
    ax2.grid(axis='y', linestyle='--', alpha=0.5)
    ax2.spines[['top','right']].set_visible(False)

    # ③ Lorenz Curve + Gini
    ax3 = fig.add_subplot(gs[1, 0]); ax3.set_facecolor(CARD)
    pc = np.array([0] + list(df['人數累計'] / 100))
    sc = np.array([0] + list(df['股數累計'] / 100))
    gini = 1 - 2 * np.trapezoid(sc, pc)
    ax3.plot([0,1],[0,1], '--', color=GRAY, linewidth=1.5, label='完全平等線')
    ax3.fill_between(pc, pc, sc, alpha=0.15, color=PURPLE)
    ax3.plot(pc, sc, color=PURPLE, linewidth=2.5, marker='o', markersize=4, label='Lorenz Curve')
    ax3.set_title(f'③ 洛倫茲曲線（Lorenz Curve）\nGini 係數 = {gini:.4f}',
                  fontsize=14, fontweight='bold', pad=10)
    ax3.set_xlabel('累計人數比例', fontsize=10)
    ax3.set_ylabel('累計持股比例', fontsize=10)
    ax3.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax3.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax3.legend(fontsize=9)
    ax3.annotate(f'Gini = {gini:.4f}\n（越接近1持股越集中）',
                 xy=(0.35, 0.1), fontsize=10, color=PURPLE,
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='#f3e5f5', alpha=0.8))
    ax3.grid(linestyle='--', alpha=0.4)
    ax3.spines[['top','right']].set_visible(False)

    # ④ 雙軸累計
    ax4 = fig.add_subplot(gs[1, 1]); ax4.set_facecolor(CARD)
    ax4.plot(x, df['人數累計'], color=BLUE, linewidth=2.5, marker='o', markersize=5, label='累計人數比例')
    ax4r = ax4.twinx()
    ax4r.plot(x, df['股數累計'], color=RED, linewidth=2.5, linestyle='--', marker='s', markersize=5, label='累計持股比例')
    ax4.set_title('④ 累計分布雙軸曲線', fontsize=14, fontweight='bold', pad=10)
    ax4.set_xticks(x); ax4.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
    ax4.set_ylabel('累計人數比例 (%)', fontsize=10, color=BLUE)
    ax4r.set_ylabel('累計持股比例 (%)', fontsize=10, color=RED)
    ax4.tick_params(axis='y', labelcolor=BLUE)
    ax4r.tick_params(axis='y', labelcolor=RED)
    l1, n1 = ax4.get_legend_handles_labels()
    l2, n2 = ax4r.get_legend_handles_labels()
    ax4.legend(l1+l2, n1+n2, fontsize=9, loc='upper left')
    ax4.grid(linestyle='--', alpha=0.4)
    ax4.spines[['top','right']].set_visible(False)

    # ⑤ 大 vs. 小股東泡泡
    ax5 = fig.add_subplot(gs[2, 0]); ax5.set_facecolor(CARD)
    n = len(df)
    groups = {
        '超小散戶\n(1-999股)':    (df.iloc[0]['人數'], df.iloc[0]['股數佔比'],  RED),
        '一般散戶\n(1K-50K股)':   (df.iloc[1:8]['人數'].sum(), df.iloc[1:8]['股數佔比'].sum(), ORANGE),
        '中型股東\n(50K-1M股)':   (df.iloc[8:14]['人數'].sum(), df.iloc[8:14]['股數佔比'].sum(), BLUE),
        '大股東\n(1M股以上)':      (df.iloc[14]['人數'], df.iloc[14]['股數佔比'], GREEN),
    }
    names_  = list(groups.keys())
    people_ = [v[0]/10000 for v in groups.values()]
    shares_ = [v[1] for v in groups.values()]
    clrs_   = [v[2] for v in groups.values()]
    ax5.scatter(people_, shares_, s=[p*1.5+200 for p in people_],
                c=clrs_, alpha=0.85, edgecolors='white', linewidth=2)
    for i, name in enumerate(names_):
        ax5.annotate(name, xy=(people_[i], shares_[i]), xytext=(10, 8),
                     textcoords='offset points', fontsize=9, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    ax5.set_title('⑤ 大 vs. 小股東對比', fontsize=14, fontweight='bold', pad=10)
    ax5.set_xlabel('人數（萬人）', fontsize=10)
    ax5.set_ylabel('持股佔比 (%)', fontsize=10)
    ax5.grid(linestyle='--', alpha=0.4)
    ax5.spines[['top','right']].set_visible(False)

    # ⑥ 平均持股橫條
    ax6 = fig.add_subplot(gs[2, 1]); ax6.set_facecolor(CARD)
    avg_log = np.log10(df['平均持股'])
    norm    = (avg_log - avg_log.min()) / (avg_log.max() - avg_log.min())
    hbars   = ax6.barh(labels, df['平均持股'],
                       color=plt.cm.RdYlGn(norm), edgecolor='white')
    ax6.set_title('⑥ 各持股區間平均持股數\n（對數尺度）', fontsize=14, fontweight='bold', pad=10)
    ax6.set_xlabel('平均持股數（股）', fontsize=10)
    ax6.set_xscale('log')
    ax6.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    ax6.tick_params(axis='y', labelsize=7)
    for bar, val in zip(hbars, df['平均持股']):
        ax6.annotate(f'{val:,.0f}',
                     xy=(val, bar.get_y()+bar.get_height()/2),
                     xytext=(5, 0), textcoords='offset points', va='center', fontsize=7.5)
    ax6.grid(axis='x', linestyle='--', alpha=0.4)
    ax6.spines[['top','right']].set_visible(False)

    fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
    #plt.close(fig)
    print(f'圖表已儲存：{out_path}')


# 主程式

In [6]:
csv_path = "https://opendata.tdcc.com.tw/getOD.ashx?id=1-5"
stock_list = ["2317", "2330", "0050", "006208", "00631L"]

In [7]:
# ── 主程式 ────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    #if len(sys.argv) < 3:
    #    print('用法：python tdcc_from_csv.py <CSV路徑> <股票代號> [日期yyyymmdd]')
    #    sys.exit(1)

    #csv_path = sys.argv[1]
    #stock_no = sys.argv[2]
    date_arg = sys.argv[3] if len(sys.argv) > 3 else None


    for stock_no in stock_list:
      try:
        df, date = load_data(csv_path, stock_no, date_arg)

        print(f'\n── {stock_no}  {date} 持股分布 ──')
        print(df[['持股區間', '人數', '人數佔比', '人數累計', '股數', '股數佔比', '股數累計']].rename(columns={
            '人數佔比': '占總人數比例(%)',
            '人數累計': '占總人數累計比例(%)',
            '股數佔比': '占集保庫存數比例(%)',
            '股數累計': '占集保庫存數累計比例(%)',
        }).to_string(index=False))

        out_file = f'{stock_no}_持股分析_{date}.png'
        plot_analysis(df, stock_no, date, out_file)

      except ValueError as e:
        print(f'⚠️  {stock_no} 跳過：{e}')

使用最新日期：20260709（可用日期：20260709…）

── 2317  20260709 持股分布 ──
             持股區間     人數  占總人數比例(%)  占總人數累計比例(%)         股數  占集保庫存數比例(%)  占集保庫存數累計比例(%)
            1-999 448573  37.899389    37.899389  102162591         0.72           0.72
      1,000-5,000 569782  48.140191    86.039580 1177044932         8.36           9.08
     5,001-10,000  85925   7.259699    93.299279  644243466         4.57          13.65
    10,001-15,000  28726   2.427025    95.726304  358835089         2.54          16.19
    15,001-20,000  14855   1.255081    96.981385  266142306         1.89          18.08
    20,001-30,000  13976   1.180815    98.162200  347682237         2.47          20.55
    30,001-40,000   6316   0.533631    98.695831  221600995         1.57          22.12
    40,001-50,000   3776   0.319030    99.014861  172056123         1.22          23.34
   50,001-100,000   6541   0.552641    99.567502  455537380         3.23          26.57
  100,001-200,000   2674   0.225923    99.793425  362940736  